# Inverse covariance estimation with MFCF-LoGo.

## Generate data

In [20]:
import numpy as np
from scipy import linalg

def generate_spd_precision(n=5, density=0.2, eps=1e-8, prng=None):
    A = prng.random(size=(n, n)) * 1000
    precision = (A + A.T) / 2.0  # make symmetric

    # Sparsify (keep zeros symmetric; don't zero the diagonal)
    mask = prng.uniform(size=(n, n)) < density
    mask = np.triu(mask, k=1)  # keep strictly upper triangle
    mask = mask + mask.T
    np.fill_diagonal(mask, False)
    precision[mask] = 0.0

    # Diagonal loading: preserve ALL off-diagonal zeros, ensure PD
    min_eig = np.linalg.eigvalsh(precision).min()
    if min_eig <= eps:
        precision += (-min_eig + eps) * np.eye(n)

    return precision

n_samples = 100
n_features = 2500
prng = np.random.RandomState(1)

prec = generate_spd_precision(n_features, density=0.8, prng=prng)
cov = linalg.inv(prec)
d = np.sqrt(np.diag(cov))
cov /= d
cov /= d[:, np.newaxis]
prec *= d
prec *= d[:, np.newaxis]

X = prng.multivariate_normal(np.zeros(n_features), cov, size=n_samples)

## Estimate the covariance and precision matrices

In [21]:
from sklearn.covariance import GraphicalLassoCV
from mfcf_logo import MFCFLoGoCV, MFCFLoGo
import time

emp_cov = np.dot(X.T, X) / n_samples

start = time.time()
model = MFCFLoGo()
model.fit(X)
end = time.time()
print(end - start)
cov_ = model.covariance_
prec_ = model.precision_

0.9668428897857666


In [22]:
prec_

array([[3.83551105e+08, 0.00000000e+00, 0.00000000e+00, ...,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 1.37578114e+08, 0.00000000e+00, ...,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 1.19170743e+09, ...,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       ...,
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
        1.35030692e+06, 0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
        0.00000000e+00, 1.85030245e+08, 0.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
        0.00000000e+00, 0.00000000e+00, 9.22155247e+08]],
      shape=(2500, 2500))

## Test Mutual Information Scenario

In [29]:
"""
Setup
-----
``n_latents`` latent factors ``Z_k ~ N(0, 1)``, i.i.d.  Each latent drives a
block of ``block_size`` features via NON-MONOTONE transforms
(``cos(2z), sin(2z), z**2 - 1, |z| - 0.8``) plus small Gaussian noise.
Ground truth: two features are conditionally dependent iff they belong to
the same block; the precision matrix's support is that block adjacency.

Pearson correlation cannot see these dependencies because every transform is
symmetric in ``z``, so ``cov(f_a(Z), f_b(Z)) ~= 0``.  The KSG MI estimator
looks at the joint density directly and recovers the block structure.
"""
import numpy as np

from mfcf_logo import MFCFLoGo


def _generate_nonmonotone_blocks(
    n: int, n_latents: int, block_size: int, seed: int
) -> tuple:
    """Latent-block data designed so MI beats correlation.

    Returns
    -------
    X : ndarray of shape (n, n_latents * block_size)
    prec_true : ndarray of shape (p, p)
        Ground-truth precision *support pattern*: 1 on the diagonal and on
        same-block off-diagonals, 0 elsewhere.  Magnitudes are not modelled
        (the data are not Gaussian); the sparsity pattern is what defines the
        conditional-independence structure of interest.
    """
    prng = np.random.RandomState(seed)
    p = n_latents * block_size
    Z = prng.randn(n, n_latents)
    fs = [
        lambda z: np.cos(2 * z),
        lambda z: np.sin(2 * z),
        lambda z: z ** 2 - 1.0,
        lambda z: np.abs(z) - 0.8,
    ]
    X = np.empty((n, p))
    true_block = np.empty(p, dtype=int)
    for k in range(n_latents):
        for b in range(block_size):
            X[:, k * block_size + b] = fs[b](Z[:, k]) + 0.05 * prng.randn(n)
            true_block[k * block_size + b] = k
    prec_true = (true_block[:, None] == true_block[None, :]).astype(float)
    return X, prec_true


def _edges_from_precision(P: np.ndarray, thr: float = 1e-8) -> set:
    """Off-diagonal support of ``P`` as a set of undirected edges ``(i, j)``."""
    A = np.abs(P) > thr
    np.fill_diagonal(A, False)
    p = P.shape[0]
    return {(i, j) for i in range(p) for j in range(i + 1, p) if A[i, j]}


def _edge_scores(E_est: set, E_true: set) -> tuple:
    """Return (precision, recall, F1) for an estimated edge set vs ground truth."""
    tp = len(E_est & E_true)
    fp = len(E_est - E_true)
    fn = len(E_true - E_est)
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-12)
    return precision, recall, f1


if __name__ == "__main__":
    n_samples, n_latents, block_size = 70, 50, 4
    n_features = n_latents * block_size  

    X, prec_true = _generate_nonmonotone_blocks(
        n=n_samples, n_latents=n_latents, block_size=block_size, seed=0
    )
    E_true = _edges_from_precision(prec_true)

    model_mi = MFCFLoGo(
        similarity="mutual_information",
        mi_n_neighbors=3,
        mi_normalize="linfoot",
        mi_random_state=10,
    ).fit(X)
    prec_hat_mi = model_mi.precision_

    model_corr = MFCFLoGo(similarity="correlation").fit(X)
    prec_hat_corr = model_corr.precision_

    E_hat_mi = _edges_from_precision(prec_hat_mi)
    E_hat_corr = _edges_from_precision(prec_hat_corr)
    p_mi, r_mi, f1_mi = _edge_scores(E_hat_mi, E_true)
    p_co, r_co, f1_co = _edge_scores(E_hat_corr, E_true)

    print(
        f"MFCF-LoGo precision recovery on non-monotone latent blocks "
        f"(n={n_samples}, p={n_features}, {n_latents} blocks of "
        f"{block_size} features each):"
    )
    with np.printoptions(precision=3, suppress=True, linewidth=160):
        print("\nOriginal (ground-truth) precision sparsity pattern "
              "(1 = same-block dependence):")
        print(prec_true)
        print("\nEstimated precision matrix (similarity='mutual_information'):")
        print(prec_hat_mi)
        print("\nEstimated precision matrix (similarity='correlation'):")
        print(prec_hat_corr)
    print(
        f"\nEdge recovery vs ground-truth block adjacency "
        f"(|E_true|={len(E_true)}):"
    )
    print(
        f"  mutual_information : precision={p_mi:.2f}  recall={r_mi:.2f}  "
        f"F1={f1_mi:.2f}  (|E_est|={len(E_hat_mi)})"
    )
    print(
        f"  correlation        : precision={p_co:.2f}  recall={r_co:.2f}  "
        f"F1={f1_co:.2f}  (|E_est|={len(E_hat_corr)})"
    )


MFCF-LoGo precision recovery on non-monotone latent blocks (n=70, p=200, 50 blocks of 4 features each):

Original (ground-truth) precision sparsity pattern (1 = same-block dependence):
[[1. 1. 1. ... 0. 0. 0.]
 [1. 1. 1. ... 0. 0. 0.]
 [1. 1. 1. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 1. 1. 1.]
 [0. 0. 0. ... 1. 1. 1.]
 [0. 0. 0. ... 1. 1. 1.]]

Estimated precision matrix (similarity='mutual_information'):
[[ 17.377  -0.564  -8.865 ...   0.      0.      0.   ]
 [ -0.564   2.505  -1.06  ...   0.      0.      0.   ]
 [ -8.865  -1.06   11.964 ...   0.      0.      0.   ]
 ...
 [  0.      0.      0.    ...   2.187  -0.456  -0.288]
 [  0.      0.      0.    ...  -0.456  15.272 -42.757]
 [  0.      0.      0.    ...  -0.288 -42.757 142.417]]

Estimated precision matrix (similarity='correlation'):
[[  9.245   0.     -9.43  ...   0.      0.      0.   ]
 [  0.      1.25    0.    ...   0.      0.      0.   ]
 [ -9.43    0.     19.869 ...   0.      0.      0.   ]
 ...
 [  0.      0.      0.    ...   1.